# Snap Media Browser for Colab

This notebook clones the repo, mounts Google Drive, installs compatible packages into an isolated target directory, prints diagnostics, and launches `snap_media_browser.py` against a Drive folder.

Run the cells from top to bottom. The last cell keeps running while the app is live.

The notebook avoids mutating Colab's global Python environment, so runtime debugging stays reproducible.

Notebook marker: `COLAB_NOTEBOOK_MODE=PKG_DIR`.

After running cell 1, confirm the output shows both:
- `COLAB_NOTEBOOK_MODE=PKG_DIR`
- a `git rev-parse --short HEAD` value from the cloned repo

In [ ]:
#@title 1. Clone the repo and install isolated packages
REPO_URL = "https://github.com/ivyking48/Gradio-ClaudeCode-Tester.git"  #@param {type:"string"}
BRANCH = "codex-fix-snap-media-browser-docs"  #@param {type:"string"}
PKG_DIR = "/content/snap-media-browser-pkgs"  #@param {type:"string"}
NOTEBOOK_MODE = "PKG_DIR"  #@param {type:"string"}

%cd /content
print("COLAB_NOTEBOOK_MODE=", NOTEBOOK_MODE)
!rm -rf Gradio-ClaudeCode-Tester "$PKG_DIR"
!git clone --branch "$BRANCH" "$REPO_URL"
%cd /content/Gradio-ClaudeCode-Tester
!git branch --show-current
!git rev-parse --short HEAD
!python3 -m pip install -q --upgrade pip setuptools wheel
!python3 -m pip install -q --target "$PKG_DIR" "gradio>=6,<7" "Pillow>=11.0,<12"
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
print("PKG_DIR=", "$PKG_DIR")

In [ ]:
#@title 2. Mount Drive and choose app settings
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

MEDIA_ROOT = "/content/drive/MyDrive/SnapInsta_Downloads"  #@param {type:"string"}
THUMB_DIR = "/content/_snap_thumbs"  #@param {type:"string"}
SHARE_PUBLIC = True  #@param {type:"boolean"}
DEBUG_LOGS = True  #@param {type:"boolean"}

os.environ["SNAP_MEDIA_ROOT"] = MEDIA_ROOT
os.environ["SNAP_THUMB_DIR"] = THUMB_DIR
os.environ["SNAP_SHARE"] = "1" if SHARE_PUBLIC else "0"
os.environ["SNAP_SKIP_COLAB_MOUNT"] = "1"
os.environ["SNAP_DEBUG"] = "1" if DEBUG_LOGS else "0"
os.environ["SNAP_EXTRA_PYTHONPATH"] = "/content/snap-media-browser-pkgs"

print("SNAP_MEDIA_ROOT=", os.environ["SNAP_MEDIA_ROOT"])
print("SNAP_THUMB_DIR=", os.environ["SNAP_THUMB_DIR"])
print("SNAP_SHARE=", os.environ["SNAP_SHARE"])
print("SNAP_SKIP_COLAB_MOUNT=", os.environ["SNAP_SKIP_COLAB_MOUNT"])
print("SNAP_DEBUG=", os.environ["SNAP_DEBUG"])
print("SNAP_EXTRA_PYTHONPATH=", os.environ["SNAP_EXTRA_PYTHONPATH"])

In [ ]:
# 3. Preflight diagnostics
import os
import pathlib
import subprocess

repo = pathlib.Path('/content/Gradio-ClaudeCode-Tester')
pkg_dir = pathlib.Path('/content/snap-media-browser-pkgs')
media_root = pathlib.Path(os.environ['SNAP_MEDIA_ROOT'])
thumb_dir = pathlib.Path(os.environ['SNAP_THUMB_DIR'])

print('repo exists:', repo.exists())
print('pkg dir exists:', pkg_dir.exists())
print('media root exists:', media_root.exists())
print('thumb dir exists:', thumb_dir.exists())

if media_root.exists():
    try:
        sample = []
        for idx, item in enumerate(media_root.iterdir()):
            sample.append(item.name)
            if idx >= 4:
                break
        print('media sample:', sample)
    except Exception as exc:
        print('media sample failed:', repr(exc))

env = os.environ.copy()
env['PYTHONPATH'] = env['SNAP_EXTRA_PYTHONPATH'] + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
code = r'''
import inspect, os, sys
import gradio as gr
from PIL import Image
print('python:', sys.version)
print('executable:', sys.executable)
print('sys.path[0:5]:', sys.path[:5])
print('gradio:', getattr(gr, '__version__', 'unknown'))
print('pillow:', getattr(Image, '__version__', 'unknown'))
print('html_signature:', inspect.signature(gr.HTML.__init__))
print('html_has_custom_api:', all(name in inspect.signature(gr.HTML.__init__).parameters for name in ['html_template', 'css_template', 'js_on_load']))
print('env_media_root:', os.environ.get('SNAP_MEDIA_ROOT'))
print('env_thumb_dir:', os.environ.get('SNAP_THUMB_DIR'))
print('env_share:', os.environ.get('SNAP_SHARE'))
print('env_skip_mount:', os.environ.get('SNAP_SKIP_COLAB_MOUNT'))
print('env_debug:', os.environ.get('SNAP_DEBUG'))
'''
subprocess.run(['python3', '-c', code], check=True, env=env)

In [ ]:
# 4. Launch the app
%cd /content/Gradio-ClaudeCode-Tester
!PYTHONUNBUFFERED=1 PYTHONPATH=/content/snap-media-browser-pkgs python3 snap_media_browser.py